In [11]:
import pandas as pd

df = pd.read_csv("RAW_scraped_data.csv")
df.head()

,Element Type,Content
0,H1,L’expert tech — du matériel informatique & éle...
1,H2,Le Black Friday dure tout novembre sur
2,H2,Le poste de travail à prix Black Friday.
3,H2,Samsung Odyssey G3 24″ à prix Black Friday.
4,H2,Les PC MSI à prix Black Friday.


In [12]:
print("Nombre des lignes doublons:", df.duplicated().sum())
df.value_counts(["Element Type", "Content"]).head(10)

Nombre des lignes doublons: 124


Element Type  Content                                
Paragraph     En stock                                   26
              Livraison partoutLivré partout au Maroc    18
              Lenovo                                     13
              Nos marques                                11
              À la une                                    9
              TP-Link                                     9
              Samsung                                     6
              En stock au magasin de Casablanca           5
              Logitech                                    4
              Eaton                                       4
Name: count, dtype: int64

In [13]:
df.loc[df["Element Type"] == "Link", "Content"].to_string(index=False)

'             WhatsApp (https://wa.me/212664104624)\n             WhatsApp (https://wa.me/212664104624)\n             WhatsApp (https://wa.me/212664104624)\nWhatsApp — conseil & commande+212 6 64 10 46 24...\nItinéraire Google Maps (https://maps.app.goo.gl...\nItinéraire Google Maps (https://maps.app.goo.gl...\n                Recrutement (https://jobs.iris.ma)\n         Vendre sur iris.ma (https://sell.iris.ma)\nWhatsAppÉcrivez-nousDiscuter (https://wa.me/212...\nOuvrir WhatsApp Web (https://wa.me/212664104624...'

In [14]:
clean_df = df.copy()

links = clean_df["Element Type"] == "Link"
parts = clean_df.loc[links, "Content"].str.extract(
    r"^(.*?)\s*\((https?://[^)]+)\)$"
)

print("Links found:", parts[1].notna().sum(), "of", links.sum())

clean_df["URL"] = pd.NA
clean_df.loc[links, "Content"] = parts[0].str.strip().to_numpy()
clean_df.loc[links, "URL"] = parts[1].to_numpy()

clean_df.loc[links, ["Element Type", "Content", "URL"]]

Links found: 10 of 10


,Element Type,Content,URL
352,Link,WhatsApp,https://wa.me/212664104624
353,Link,WhatsApp,https://wa.me/212664104624
354,Link,WhatsApp,https://wa.me/212664104624
355,Link,WhatsApp — conseil & commande+212 6 64 10 46 24,https://wa.me/212664104624
356,Link,Itinéraire Google Maps,https://maps.app.goo.gl/gZiNA87V3dAJz9vU8
357,Link,Itinéraire Google Maps,https://maps.app.goo.gl/DdTZZKRL83L3X9UP6
358,Link,Recrutement,https://jobs.iris.ma
359,Link,Vendre sur iris.ma,https://sell.iris.ma
360,Link,WhatsAppÉcrivez-nousDiscuter,https://wa.me/212664104624?text=Bonjour%2C%0Ah...
361,Link,Ouvrir WhatsApp Web,https://wa.me/212664104624?text=Bonjour%2C%0Ah...


In [15]:
avant = clean_df["Content"].copy()

clean_df["Content"] = (
    clean_df["Content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Textes modifiés :", (avant != clean_df["Content"]).sum())
print("Textes vides :", clean_df["Content"].eq("").sum())
print("Nombre de lignes :", len(clean_df))

Textes modifiés : 22
Textes vides : 0
Nombre de lignes : 362


In [16]:
liens = clean_df["Element Type"] == "Link"

print("Liens sans URL :", clean_df.loc[liens, "URL"].isna().sum())
print("URL hors des liens :", clean_df.loc[~liens, "URL"].notna().sum())
print("Lignes identiques :", clean_df.duplicated().sum())

Liens sans URL : 0
URL hors des liens : 0
Lignes identiques : 124


In [17]:
frequences_df = (
    clean_df
    .groupby(["Element Type", "Content", "URL"], dropna=False)
    .size()
    .reset_index(name="Occurrences")
)

print("Lignes distinctes :", len(frequences_df))
print("Occurrences totales :", frequences_df["Occurrences"].sum())

frequences_df.sort_values("Occurrences", ascending=False).head(10)

Lignes distinctes : 238
Occurrences totales : 362


,Element Type,Content,URL,Occurrences
121,Paragraph,En stock,NaN,26
149,Paragraph,Livraison partoutLivré partout au Maroc,NaN,18
148,Paragraph,Lenovo,NaN,13
160,Paragraph,Nos marques,NaN,11
205,Paragraph,TP-Link,NaN,9
222,Paragraph,À la une,NaN,9
199,Paragraph,Samsung,NaN,6
122,Paragraph,En stock au magasin de Casablanca,NaN,5
83,Paragraph,Black November,NaN,4
210,Paragraph,Ugreen,NaN,4


In [18]:
assert frequences_df["Occurrences"].sum() == len(clean_df)

clean_df.to_csv("cleaned_data.csv", index=False, encoding="utf-8-sig")
frequences_df.to_csv("content_frequencies.csv", index=False, encoding="utf-8-sig")

print("Fichiers enregistrés : cleaned_data.csv et content_frequencies.csv")
print("Lignes du fichier nettoyé :", len(clean_df))
print("Lignes du tableau de fréquences :", len(frequences_df))

Fichiers enregistrés : cleaned_data.csv et content_frequencies.csv
Lignes du fichier nettoyé : 362
Lignes du tableau de fréquences : 238
